# Pupil size per trial mode

In [1]:
prefix = '/home/ines/repositories/'
# prefix = '/Users/ineslaranjeira/Documents/Repositories/'

In [9]:
""" 
IMPORTS
"""
import os
import numpy as np
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# --Machine learning and statistics

import logging
import sklearn.linear_model as sklm
from brainbox.io.one import SessionLoader
from brainbox.behavior.dlc import get_licks
import logging
from pathlib import Path
import sklearn.linear_model as sklm
from iblutil.numerical import bincount2D

# Get my functions
functions_path =  prefix + 'representation_learning_variability/Models/Sub-trial//2_fit_models/'
os.chdir(functions_path)
from preprocessing_functions import idxs_from_files
functions_path =  prefix + 'representation_learning_variability/Models/Sub-trial/4_analyses/5_clustering_analyses/'
os.chdir(functions_path)
# from clustering_functions import calculate_entropy
functions_path =  prefix + 'representation_learning_variability/Models/Sub-trial//3_postprocess_results/'
os.chdir(functions_path)
from plotting_functions import create_grouped_gradient_palette
from one.api import ONE
one = ONE(mode='remote')

In [3]:
data_path = prefix + 'representation_learning_variability/paper-individuality/fig1_segmentation/'
states_file = pd.read_parquet(data_path+'states_trial_type_09-29-2025')

data_path = prefix + 'representation_learning_variability/paper-individuality/fig1_segmentation/'
filename = str(data_path + 'all_sequences_09-29-2025')
all_sequences = pd.read_parquet(filename)

FileNotFoundError: [Errno 2] No such file or directory: '/home/ines/repositories/representation_learning_variability/paper-individuality/mixed_models/metadata_01-11-2026'

In [103]:
path = prefix + 'representation_learning_variability/paper-individuality/clustering/'
# all_sequences = pd.read_parquet(seq_path+'all_sequences_09-22-2025')
# trial_modes = pd.read_parquet(path+'trial_clusters')
trial_modes = pd.read_parquet(path+'9_cluster_per_trial')
trial_modes = pd.read_parquet(path+'17_trial_waterclust')

In [104]:
all_sequences['trial_id'] = all_sequences['sample'].str.split().str[1:2].str.join('').astype(float)
all_sequences['session'] = all_sequences['sample'].str.split().str[0:1].str.join('').astype(str)

In [105]:
all_sequences = all_sequences.merge(trial_modes[['sample', 'trial_cluster']], on='sample')

In [106]:
from scipy.signal import butter, lfilter
from scipy.stats import median_abs_deviation

def compute_pupil_psth(sl, condition_col, trial_clusters):
    
    df = sl.pose['leftCamera'].copy()
    trials = sl.trials.copy()
    trials['contrastLeft'] = trials['contrastLeft'].fillna(0)
    trials['contrastRight'] = trials['contrastRight'].fillna(0)
    trials['contrast'] = trials['contrastLeft'] + trials['contrastRight']   
    trials['trial_cluster'] = trial_clusters['trial_cluster']
    
    # --- PARAMETERS ---
    t_pre, t_post = -.6, 4.0
    baseline_win = (-0.3, -0)
    lowpass_hz = 4
    smooth_win_s = 0.05
    # condition_col = 'contrast'  # <-- change if needed

    # --- SAMPLING ---
    times = df['times'].values
    fs = 1/np.median(np.diff(times))

    # --- DIAMETER ---
    vert = np.sqrt((df['pupil_top_r_x']-df['pupil_bottom_r_x'])**2 + (df['pupil_top_r_y']-df['pupil_bottom_r_y'])**2)
    horiz = np.sqrt((df['pupil_left_r_x']-df['pupil_right_r_x'])**2 + (df['pupil_left_r_y']-df['pupil_right_r_y'])**2)
    diam = ((vert+horiz)/2).values

    # --- OUTLIERS (MAD) ---
    med = np.nanmedian(diam)
    mad = median_abs_deviation(diam,nan_policy='omit')
    diam[np.abs(diam-med)>5*mad] = np.nan
    diam = pd.Series(diam).interpolate(limit_direction='both').values

    # --- CAUSAL LOWPASS ---
    b,a = butter(2,lowpass_hz/(fs/2),btype='low')
    diam_filt = lfilter(b,a,diam)

    # --- NORMALIZE TO % SESSION MAX ---
    session_max = np.nanpercentile(diam_filt,99)
    diam_norm = 100*diam_filt/session_max

    # --- ALIGN + CONDITION SPLIT ---
    stim_times = trials['stimOn_times'].values
    conditions = trials[condition_col].values
    unique_conds = np.unique(conditions)
    t_axis = np.arange(t_pre,t_post,1/fs)
    smooth_win = int(smooth_win_s*fs)

    psth_dict = {c:[] for c in unique_conds}

    for st,cond in zip(stim_times,conditions):
        rel_time = times - st
        mask = (rel_time>=t_pre)&(rel_time<=t_post)
        if np.sum(mask)<10: continue
        trial_trace = diam_norm[mask]
        trial_time = rel_time[mask]
        baseline_mask = (trial_time>=baseline_win[0])&(trial_time<=baseline_win[1])
        if np.sum(baseline_mask)==0: continue
        trial_trace = trial_trace - np.mean(trial_trace[baseline_mask])
        trial_interp = np.interp(t_axis,trial_time,trial_trace)
        trial_smooth = np.convolve(trial_interp,np.ones(smooth_win)/smooth_win,mode='full')[:len(trial_interp)]
        psth_dict[cond].append(trial_smooth)
        
        
    palette = sns.color_palette('tab20', len(unique_conds), as_cmap=False)
    color_map = {cond:palette[i%len(palette)] for i,cond in enumerate(unique_conds)}
    # --- PLOT ---
    plt.figure(figsize=(7,4))
    for cond in unique_conds:
        data = np.array(psth_dict[cond])
        mean = np.nanmean(data,axis=0)
        sem = np.nanstd(data,axis=0)/np.sqrt(len(data))
        c = color_map[cond]
        plt.plot(t_axis,mean, color=c, label=str(cond))
        plt.fill_between(t_axis,mean-sem,mean+sem,color=c, alpha=0.25)

    plt.axvline(0,color='k',linestyle='--')
    plt.xlabel('Time from stim onset (s)')
    plt.ylabel('Pupil diameter (% max, baseline subtracted)')
    plt.legend(title=condition_col)
    plt.tight_layout()
    plt.show()

In [107]:
sessions = all_sequences['session'].unique()
condition = 'trial_cluster'
for s, session in enumerate(sessions):
    # try:
    sl = SessionLoader(eid=session, one=one)
    sl.load_pose(views=['left'], tracker='lightningPose')
    sl.load_session_data(trials=True, wheel=False, motion_energy=False)
    trial_clusters = all_sequences.loc[all_sequences['session']==session]
    trial_clusters = trial_clusters.sort_values(by='trial_id')
    compute_pupil_psth(sl, condition, trial_clusters)
    # except:
    #     print(session)
    

/home/ines/miniconda3/envs/iblenv/lib/python3.10/site-packages/one/util.py:436: ALFWarning: Multiple revisions: "2025-05-30", ""
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)


2026-03-02 18:43:11 INFO     one.py:1288 Loading trials data


/home/ines/miniconda3/envs/iblenv/lib/python3.10/site-packages/one/util.py:436: ALFWarning: Multiple revisions: "", "2025-03-03"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)


2026-03-02 18:43:12 INFO     one.py:1288 Loading pupil data


/home/ines/miniconda3/envs/iblenv/lib/python3.10/site-packages/one/util.py:436: ALFWarning: Multiple revisions: "2025-06-20", ""
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)


...

6f09ba7e-e3ce-44b0-932b-c003fb44fb89
2026-03-02 18:43:48 INFO     one.py:1288 Loading trials data


/home/ines/miniconda3/envs/iblenv/lib/python3.10/site-packages/one/util.py:436: ALFWarning: Multiple revisions: "", "2025-03-03"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)


2026-03-02 18:43:48 INFO     one.py:1288 Loading pupil data
2026-03-02 18:43:48 WARNING  one.py:1292 Could not load pupil data.


/home/ines/miniconda3/envs/iblenv/lib/python3.10/site-packages/one/util.py:436: ALFWarning: Multiple revisions: "2025-06-04", ""
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)


e56541a5-a6d5-4750-b1fe-f6b5257bfe7c
2026-03-02 18:43:53 INFO     one.py:1288 Loading trials data
